# Walk-Forward Forecasting with LightGBM for Production

## Libraries

In [1]:
import numpy as np
import pandas as pd

## Data Loading

In [2]:
train = pd.read_csv('../data/raw/train.csv', parse_dates=['Date'])
store = pd.read_csv('../data/raw/store.csv')

df = train.merge(store, on='Store', how='left')
df = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

print(f"Shape : {df.shape}")
print(f"Nb stores : {df['Store'].nunique()}")
print(f"Period : {df['Date'].min()} → {df['Date'].max()}")

/var/folders/6n/y9swsjwd5w3_20bzq091vx980000gn/T/ipykernel_12648/1955849004.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../data/raw/train.csv', parse_dates=['Date'])


Shape : (844338, 18)
Nb stores : 1115
Period : 2013-01-01 00:00:00 → 2015-07-31 00:00:00


## Feature Engineering

### Averages by Window

In [53]:
def build_asof_features(df, windows=(7, 28, 91)):
    """
    Aggregate sales for a given windows ending at the current date
    """
    # sorting by stores and dates
    d = df.sort_values(["Store", "Date"]).copy()
    # days closed must not impact the averages
    d["SalesOpen"] = d["Sales"].where(d["Open"] == 1)
    g = d.groupby("Store")["SalesOpen"]
    # each row of g is a store object containing all sales data

    out = d[["Store", "Date"]].copy()
    for w in windows:
        mp = max(3, w // 3)
        out[f"asof_mean_{w}d"] = g.transform(lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).mean())
        out[f"asof_std_{w}d"]  = g.transform(lambda s, w=w, mp=mp: s.rolling(w, min_periods=mp).std())
    
    out["asof_ratio_7_28"] = out["asof_mean_7d"] / out["asof_mean_28d"]
    out["asof_ratio_28_91"] = out["asof_mean_28d"] / out["asof_mean_91d"]
    out["asof_cv_28"] = out["asof_std_28d"]  / out["asof_mean_28d"]

    return out

### Origin and Target Combinations

In [67]:
def snapshot_at(asof, ori, cols):
    """
    Last known state of each store at the origin date (included)
    This to avoid removing stores just because they were close at origin
    """
    s = asof[asof["Date"] <= ori]
    s = s = s.drop_duplicates("Store", keep="last")
    s = s.rename(columns={"Date": "snap_date"})
    s["days_since_snap"] = (ori - s["snap_date"]).dt.days
    return s[["Store", "days_since_snap"] + cols]


def make_training_frame(df, origins, horizon=42, asof=None):
    if asof is None:
        asof = build_asof_features(df)
    asof_cols = [c for c in asof.columns if c.startswith("asof_")]

    # sales from the previous year anchored on target date
    # 364 days or 52 weeks, same day of week
    # 364 > horizon so always observed at origin
    ly = df[["Store", "Date", "Sales"]].rename(columns={"Sales": "sales_ly"})
    ly["Date"] = ly["Date"] + pd.Timedelta(days=364)

    frames = []
    for ori in pd.to_datetime(sorted(origins)):
        tgt = df[(df["Date"] > ori) & (df["Date"] <= ori + pd.Timedelta(days=horizon))].copy()
        snap = snapshot_at(asof, ori, asof_cols)
        tgt = tgt.merge(snap, on="Store", how="inner")
        tgt = tgt.merge(ly, on=["Store", "Date"], how="left")
        tgt["origin"] = ori
        tgt["h"] = (tgt["Date"] - ori).dt.days
        tgt["h_week"] = np.ceil(tgt["h"] / 7).astype(int)
        frames.append(tgt)

    if not frames:
        raise ValueError(f"no workable origin into {list(origins)}")
    out = pd.concat(frames, ignore_index=True)
    return out[out["Open"] == 1].reset_index(drop=True)

### Origins and Folds

In [68]:
HORIZON = 42

candidate_origins = pd.date_range(
    start=df["Date"].min() + pd.Timedelta(days=400),
    end=df["Date"].max() - pd.Timedelta(days=HORIZON),
    freq="14D")

test_origins = pd.date_range(
    end=df["Date"].max() - pd.Timedelta(days=HORIZON), periods=6, freq="MS"
)

asof = build_asof_features(df).sort_values(["Store", "Date"]).reset_index(drop=True) # computed once for all folds

def make_fold(ori, margin=True):
    cutoff = ori - pd.Timedelta(days=HORIZON) if margin else ori
    tr_origins = candidate_origins[candidate_origins < cutoff]
    tr = make_training_frame(df, tr_origins, HORIZON, asof)
    tr = tr[tr["Date"] < ori]
    te = make_training_frame(df, [ori], HORIZON, asof)
    assert tr["Date"].max() < ori < te["Date"].min(), f"overflow on {ori.date()}"
    return tr, te

tr, te = make_fold(test_origins[0])
print(tr.shape, te.shape, tr["Date"].max(), te["Date"].min())

(746294, 32) (39923, 32) 2014-12-24 00:00:00 2015-01-02 00:00:00


In [70]:
for ori in test_origins:
    tr, te = make_fold(ori)
    print(ori.date(), len(tr), len(te), tr["Date"].max().date(), te["Date"].min().date())

2015-01-01 746294 39923 2014-12-24 2015-01-02
2015-02-01 811698 40170 2015-01-21 2015-02-02
2015-03-01 887804 38090 2015-02-18 2015-03-02
2015-04-01 968138 37040 2015-03-18 2015-04-02
2015-05-01 1084595 37472 2015-04-29 2015-05-02
2015-06-01 1158695 39626 2015-05-27 2015-06-02
